In [ ]:
# ============================================================================
# 1. IMPORTS
# ============================================================================

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.tools import tool
from deepagents.backends import FilesystemBackend

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✓ All imports loaded successfully")

✓ All imports loaded successfully


In [2]:
# ============================================================================
# # 2. CONFIGURATION
# ============================================================================

# Project paths - auto-detect from current working directory
PROJECT_ROOT = Path.cwd() / ".."
WORK_DIR = PROJECT_ROOT / "work"
DOCS_DIR = PROJECT_ROOT / "docs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"

# Model configuration
EMBEDDING_MODEL = "nomic-embed-text"
REVIEW_MODEL_NAME = "qwen2.5-coder:7b-instruct"
AGENT_MODEL = "claude-haiku-4-5-20251001"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Work Dir: {WORK_DIR}")
print(f"Docs Dir: {DOCS_DIR}")

# ============================================================================
# 3. PROMPT LOADER UTILITY
# ============================================================================

def load_prompt(filename: str) -> str:
    """Load a prompt template from a markdown file."""
    filepath = PROMPTS_DIR / filename
    if not filepath.exists():
        raise FileNotFoundError(f"Prompt file not found: {filepath}")
    return filepath.read_text()

# Pre-load all prompts
print("\n📂 Loading prompt templates...")
VALIDATE_PROMPT_TEMPLATE = load_prompt("terraform-validate.md")
REVIEW_PROMPT_TEMPLATE = load_prompt("terraform-review.md")
SYSTEM_PROMPT = load_prompt("terraform-system.md")
USER_PROMPT = load_prompt("terraform-user.md")
REVIEW_RESPONSE_WITH_FIXES = load_prompt("review-response-with-fixes.md")
REVIEW_RESPONSE_COMPLIANT = load_prompt("review-response-compliant.md")
print(f"  ✓ Loaded validation prompt template")
print(f"  ✓ Loaded review prompt template")
print(f"  ✓ Loaded system prompt template")
print(f"  ✓ Loaded user prompt template")
print(f"  ✓ Loaded review response templates")


# ============================================================================
# 5. Création de la base de données et indexation des documents
# ============================================================================

print("\n📚 Loading knowledge base...")

# Load all documents from the docs directory
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    recursive=True
)
documents = loader.load()
print(f"  ✓ Loaded {len(documents)} document(s) from {DOCS_DIR}")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
docs = text_splitter.split_documents(documents)
print(f"  ✓ Split into {len(docs)} chunks")


# Initialize embedding function
embedding_function = OllamaEmbeddings(model=EMBEDDING_MODEL)

# Step 1: Create the vectorstore database
print(f"  Creating vectorstore database...")
vectorstore = Chroma(
    collection_name="terraform_docs",
    embedding_function=embedding_function, 
    persist_directory=str(PROJECT_ROOT / "notebooks" / ".vectorstore2")
)

# Step 2: Add documents to the vectorstore
print(f"  Indexing {len(docs)} documents...")
vectorstore.add_documents(docs)

print(f"  ✓ Vectorstore created and indexed")

Project Root: /Users/melkouhen/audit-tools/test-langchain/notebooks/..
Work Dir: /Users/melkouhen/audit-tools/test-langchain/notebooks/../work
Docs Dir: /Users/melkouhen/audit-tools/test-langchain/notebooks/../docs

📂 Loading prompt templates...
  ✓ Loaded validation prompt template
  ✓ Loaded review prompt template
  ✓ Loaded system prompt template
  ✓ Loaded user prompt template
  ✓ Loaded review response templates

📚 Loading knowledge base...
  ✓ Loaded 2 document(s) from /Users/melkouhen/audit-tools/test-langchain/notebooks/../docs
  ✓ Split into 20 chunks
  Creating vectorstore database...
  Indexing 20 documents...
  ✓ Vectorstore created and indexed


In [ ]:
# ============================================================================
# 7. INITIALIZE REVIEW MODEL (BEFORE tools that use it)
# ============================================================================

print("\n🤖 Initializing review model...")
review_model = ChatOllama(model=REVIEW_MODEL_NAME)
print(f"  ✓ Review model initialized: {REVIEW_MODEL_NAME}")

# ============================================================================
# 8. TOOL: Validate and Fix Code
# ============================================================================

@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for relevant Terraform best practices and standards.
    Returns the top 3 most relevant documents matching the query.
    """
    results = vectorstore.similarity_search(query, k=3)
    return "\n---\n".join([res.page_content for res in results])

@tool
def validate_and_fix_code(path: str) -> str:
    """
    Executes terraform validate on generated code.
    
    Args: 
        path : folder where the code is generated
    """
    try:
        result = subprocess.run(
            ["terraform", "validate"],
            cwd=path,
            capture_output=True,
            text=True,
            timeout=30
        )

        if result.returncode != 0:
            error_message = result.stderr or result.stdout
            
            # Use template from markdown file
            prompt = VALIDATE_PROMPT_TEMPLATE.format(
                error_message=error_message,
                root_folder=path
            )

            correction = review_model.invoke(prompt).content
            return f"❌ Erreurs détectées:\n{error_message}\n\n💡 Corrections suggérées:\n{correction}"
        else:
            return "✅ Terraform validate: succès - aucune erreur détectée"

    except FileNotFoundError:
        return "⚠️ Erreur: terraform n'est pas installé ou non accessible dans le PATH"
    except subprocess.TimeoutExpired:
        return "⚠️ Erreur: terraform validate a dépassé le timeout (30s)"
    except Exception as e:
        return f"⚠️ Erreur lors de la validation: {str(e)}"


# ============================================================================
# 8b. TOOL: Code Review & Best Practices Enforcement
# ============================================================================

@tool
def review_and_fix_code(path: str) -> str:
    """
    Performs comprehensive code review against best practices.
    
    Process:
    1. Retrieves Terraform best practices from knowledge base
    2. Analyzes generated code for compliance
    3. Identifies major issues and applies fixes if necessary

    Args: 
        path : folder where the code is generated
    """
    import glob
    
    try:
        # Step 1: Retrieve best practices from knowledge base
        best_practices = search_knowledge_base(
            "Terraform best practices security standards naming conventions modules"
        )
        
        # Step 2: Read all generated Terraform files
        tf_files = sorted(glob.glob(path + "/**/*.tf", recursive=True))
        
        if not tf_files:
            return "⚠️ Revue: Aucun fichier .tf trouvé dans le répertoire"
        
        code_content = ""
        for file_path in tf_files:
            with open(file_path, 'r') as f:
                file_name = os.path.basename(file_path)
                code_content += f"\n\n--- {file_name} ---\n{f.read()}"
        
        # Step 3: Use template from markdown file
        review_prompt = REVIEW_PROMPT_TEMPLATE.format(
            best_practices=best_practices,
            code_content=code_content
        )
        
        # Step 4: Execute review with model
        review_response = review_model.invoke(review_prompt).content
        
        # Step 5: Parse response and apply fixes if needed
        if "CRITIQUE" in review_response or "MAJEUR" in review_response:
            # Extract corrected code from response
            if "### Code Corrigé" in review_response:
                result_summary = REVIEW_RESPONSE_WITH_FIXES.format(
                    num_files=len(tf_files),
                    review_response=review_response,
                    root_folder=path
                )
            else:
                result_summary = review_response
        else:
            result_summary = REVIEW_RESPONSE_COMPLIANT.format(
                num_files=len(tf_files),
                review_response=review_response
            )
        
        return result_summary

    except FileNotFoundError as e:
        return f"⚠️ Erreur: Fichier non trouvé: {e}"
    except Exception as e:
        return f"⚠️ Erreur lors de la revue: {str(e)}"


# ============================================================================
# 9. AGENT SETUP
# ============================================================================

print("\n🤖 Setting up agent...")

# Create agent with tools
backend = FilesystemBackend(root_dir=PROJECT_ROOT, virtual_mode=False)
code_generation_agent = create_deep_agent(
    model=init_chat_model(model=AGENT_MODEL),
    backend=backend,
    tools=[search_knowledge_base, validate_and_fix_code, review_and_fix_code]
)
print(f"  ✓ System prompt loaded ({len(SYSTEM_PROMPT)} chars)")
print(f"  ✓ User prompt loaded ({len(USER_PROMPT)} chars)")
print(f"  ✓ Agent created with tools:")
print(f"    - search_knowledge_base")
print(f"    - validate_and_fix_code")
print(f"    - review_and_fix_code")


🤖 Initializing review model...
  ✓ Review model initialized: qwen2.5-coder:7b-instruct

🤖 Setting up agent...
  ✓ System prompt loaded (2328 chars)
  ✓ User prompt loaded (416 chars)
  ✓ Agent created with tools:
    - search_knowledge_base
    - validate_and_fix_code
    - review_and_fix_code


In [ ]:
# ============================================================================
# 8. PREPARE WORKSPACE
# ============================================================================

print("\n🛠️  Preparing workspace...")

# Clean and create work directory
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
    print(f"  ✓ Cleaned existing work directory")

WORK_DIR.mkdir(exist_ok=True)
print(f"  ✓ Created fresh work directory: {WORK_DIR}")

# ============================================================================
# 9. EXECUTE AGENT
# ============================================================================

print("\n🚀 Starting Terraform Code Generation Agent")
print("=" * 80)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

print("\n📝 Agent is running...")
print("    (Agent will autonomously call: search_knowledge_base, validate_and_fix_code, review_and_fix_code)")
print("-" * 80)

try:
    result = code_generation_agent.invoke({
        "messages": [{"role": "user", "content": USER_PROMPT}]
    })
    
    agent_output = result["messages"][-1].content
    overall_status = "✅ SUCCESS"
    
except Exception as e:
    overall_status = "❌ FAILED"
    agent_output = str(e)
    print(f"\n❌ Agent Error: {e}")
    import traceback
    traceback.print_exc()

print(overall_status)
print("-" * 80)


🛠️  Preparing workspace...
  ✓ Cleaned existing work directory
  ✓ Created fresh work directory: /Users/melkouhen/audit-tools/test-langchain/notebooks/../work

🚀 Starting Terraform Code Generation Agent
Timestamp: 2026-05-06 16:28:01

📝 Agent is running...
    (Agent will autonomously call: search_knowledge_base, validate_and_fix_code, review_and_fix_code)
--------------------------------------------------------------------------------
